# Statistical Feature Research

This notebook develops an independent statistical research branch for GC/MGC. This first increment is deliberately limited to research-environment configuration and validation of the trusted active-contract table. It does not load raw DBN data or POI outputs, rebuild upstream pipelines, construct observations or labels, or engineer features.

# 1.0 Configuration and Research Environment

This section establishes a reproducible project environment and locks the market-time conventions that later research must obey. All paths are resolved relative to the project root, UTC timestamps remain reference keys, and New York time is authoritative for every trading-session rule.

## 1.1 Imports

Only standard numerical, tabular, timezone, and Parquet dependencies are imported at this stage. Package versions and the running executable are printed so that the executed notebook records its research environment.

In [1]:
from __future__ import annotations

import random
import re
import sys
from datetime import time
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

print(f"Python {sys.version.split()[0]}")
print(f"Executable: {sys.executable}")
print(f"pandas {pd.__version__} | numpy {np.__version__} | pyarrow {pa.__version__}")

Python 3.14.5
Executable: C:\Users\abond\Desktop\WORK FILES\Systemic\Project 1\.venv-1\Scripts\python.exe
pandas 3.0.3 | numpy 2.4.6 | pyarrow 24.0.0


## 1.2 Paths

The project root is discovered by walking upward from the current working directory until both trusted project markers are found. The notebook therefore works whether launched from the project root or a notebook subdirectory. The active interpreter is required to be the project's `.venv-1` environment.

In [2]:
TRUSTED_DATA_RELATIVE_PATH = Path("data/processed/research_bars_gc_mgc_1m.parquet")
CONTEXT_RELATIVE_PATH = Path("project_docs/statistical_feature_research_context_report.md")
NOTEBOOK_RELATIVE_PATH = Path("notebooks/exploration/statistical_feature_research.ipynb")


def find_project_root(start: Path | None = None) -> Path:
    """Locate the project without relying on the notebook's launch directory."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / TRUSTED_DATA_RELATIVE_PATH).is_file() and (candidate / CONTEXT_RELATIVE_PATH).is_file():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the current working directory.")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / TRUSTED_DATA_RELATIVE_PATH
CONTEXT_REPORT_PATH = PROJECT_ROOT / CONTEXT_RELATIVE_PATH
NOTEBOOK_PATH = PROJECT_ROOT / NOTEBOOK_RELATIVE_PATH
EXPECTED_PYTHON = PROJECT_ROOT / ".venv-1" / "Scripts" / "python.exe"

assert DATA_PATH.is_file(), f"Missing trusted dataset: {DATA_PATH}"
assert CONTEXT_REPORT_PATH.is_file(), f"Missing governing report: {CONTEXT_REPORT_PATH}"
assert DATA_PATH.name == "research_bars_gc_mgc_1m.parquet" and DATA_PATH.suffix == ".parquet"
assert not any(token in DATA_PATH.name.lower() for token in (".dbn", "section6_", "section7_", "poi"))
assert Path(sys.executable).resolve() == EXPECTED_PYTHON.resolve(), (
    f"Wrong interpreter. Expected {EXPECTED_PYTHON}, received {sys.executable}."
)

environment_check = pd.Series(
    {
        "project_root": str(PROJECT_ROOT),
        "trusted_data": str(DATA_PATH.relative_to(PROJECT_ROOT)),
        "governing_context": str(CONTEXT_REPORT_PATH.relative_to(PROJECT_ROOT)),
        "active_interpreter": str(Path(sys.executable).resolve()),
        "expected_environment": ".venv-1",
        "environment_valid": True,
    },
    name="value",
)
display(environment_check.to_frame())

,value
project_root,C:\Users\abond\Desktop\WORK FILES\Systemic\Pro...
trusted_data,data\processed\research_bars_gc_mgc_1m.parquet
governing_context,project_docs\statistical_feature_research_cont...
active_interpreter,C:\Users\abond\Desktop\WORK FILES\Systemic\Pro...
expected_environment,.venv-1
environment_valid,True


## 1.3 Instrument and Time Settings

GC and MGC are the only expected products, with GC designated for the later Phase 1 discovery population. `America/New_York` is the authoritative timezone for session classification, trading dates, entry eligibility, the noon cutoff, and the 3:30pm mandatory exit. UTC is retained only as an immutable timestamp reference. The entry intervals are recorded now but are not applied until the eligible observation frame is constructed in a later section.

In [3]:
EXPECTED_PRODUCTS = ("GC", "MGC")
PRIMARY_RESEARCH_PRODUCT = "GC"
RESEARCH_TIMEZONE_NAME = "America/New_York"
REFERENCE_TIMEZONE_NAME = "UTC"
RESEARCH_TIMEZONE = ZoneInfo(RESEARCH_TIMEZONE_NAME)
REFERENCE_TIMEZONE = ZoneInfo(REFERENCE_TIMEZONE_NAME)
UTC_REFERENCE_KEY_COLUMNS = ("ts_event_utc", "product")

ENTRY_WINDOWS_NY = {
    "London": (time(3, 0), time(6, 0)),
    "New York": (time(7, 0), time(12, 0)),
}
NO_NEW_ENTRY_AT_OR_AFTER_NY = time(12, 0)
MANDATORY_EXIT_NY = time(15, 30)

assert PRIMARY_RESEARCH_PRODUCT in EXPECTED_PRODUCTS
assert RESEARCH_TIMEZONE.key == "America/New_York"
assert ENTRY_WINDOWS_NY["London"] == (time(3, 0), time(6, 0))
assert ENTRY_WINDOWS_NY["New York"] == (time(7, 0), time(12, 0))
assert NO_NEW_ENTRY_AT_OR_AFTER_NY == time(12, 0)
assert MANDATORY_EXIT_NY == time(15, 30)

time_settings = pd.Series(
    {
        "expected_products": EXPECTED_PRODUCTS,
        "phase_1_product": PRIMARY_RESEARCH_PRODUCT,
        "research_timezone": RESEARCH_TIMEZONE_NAME,
        "UTC_reference_keys": UTC_REFERENCE_KEY_COLUMNS,
        "London_entry_window_NY": "[03:00, 06:00)",
        "New_York_entry_window_NY": "[07:00, 12:00)",
        "mandatory_exit_NY": "15:30",
    },
    name="setting",
)
display(time_settings.to_frame())

,setting
expected_products,"(GC, MGC)"
phase_1_product,GC
research_timezone,America/New_York
UTC_reference_keys,"(ts_event_utc, product)"
London_entry_window_NY,"[03:00, 06:00)"
New_York_entry_window_NY,"[07:00, 12:00)"
mandatory_exit_NY,15:30


## 1.4 Random Seeds and Display Settings

A fixed seed is set for reproducible sampling and later statistical procedures. Display settings expose enough rows and columns to audit the saved schema and validation tables without changing analytical results.

In [4]:
RANDOM_SEED = 20260714
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

print(f"Random seed fixed at {RANDOM_SEED}.")

Random seed fixed at 20260714.


# 2.0 Load Trusted Research Data

The sole market-data input is the completed active-contract research table. The saved Arrow schema is inspected before the table is materialized; no raw DBN archive, upstream active-contract logic, or Section 6/7 POI output is read.

## 2.1 Load Final GC/MGC Research Table

Parquet metadata and the physical Arrow schema are read first. The complete table is then loaded once with Arrow-backed pandas dtypes to reduce memory overhead while preserving the saved timezone and integer types. Subsequent checks reuse this single dataframe.

In [5]:
parquet_file = pq.ParquetFile(DATA_PATH)
saved_arrow_schema = parquet_file.schema_arrow
saved_column_names = tuple(saved_arrow_schema.names)
parquet_metadata = pd.Series(
    {
        "relative_path": str(DATA_PATH.relative_to(PROJECT_ROOT)),
        "rows": parquet_file.metadata.num_rows,
        "columns": parquet_file.metadata.num_columns,
        "row_groups": parquet_file.metadata.num_row_groups,
        "file_size_MiB": DATA_PATH.stat().st_size / (1024**2),
    },
    name="saved_value",
)
display(parquet_metadata.to_frame())

research_bars = pd.read_parquet(DATA_PATH, engine="pyarrow", dtype_backend="pyarrow")

assert len(research_bars) == parquet_file.metadata.num_rows
assert tuple(research_bars.columns) == saved_column_names
loaded_summary = pd.Series(
    {
        "loaded_rows": len(research_bars),
        "loaded_columns": research_bars.shape[1],
        "dataframe_memory_MiB": research_bars.memory_usage(index=True, deep=True).sum() / (1024**2),
        "dtype_backend": "pyarrow",
    },
    name="loaded_value",
)
display(loaded_summary.to_frame())

,saved_value
relative_path,data\processed\research_bars_gc_mgc_1m.parquet
rows,3487656
columns,82
row_groups,4
file_size_MiB,603.389116


,loaded_value
loaded_rows,3487656
loaded_columns,82
dataframe_memory_MiB,"1,890.141650"
dtype_backend,pyarrow


## 2.2 Confirm Schema and Data Types

The saved Arrow type and loaded pandas type are shown for every column before any downstream assumptions are made. Required identifiers, market fields, session fields, and controls are asserted. Product counts, within-product timestamp order, UTC-key uniqueness, and core OHLCV completeness are validated.

Existing columns whose names indicate a future market outcome are placed in `EXISTING_FORWARD_COLUMNS`. This quarantine is a namespace safeguard: these columns remain available only for possible later validation and must never enter the independent feature frame. Operational schedule fields such as `days_to_next_roll` are rollover controls, not realized market outcomes, and are not classified as feature labels here. Timestamp sorting is required within each product; no assumption is made about the panel's physical interleaving across products.

In [6]:
schema_table = pd.DataFrame(
    {
        "column": saved_column_names,
        "saved_arrow_type": [str(field.type) for field in saved_arrow_schema],
        "loaded_pandas_type": [str(research_bars.dtypes[name]) for name in saved_column_names],
    }
)
display(schema_table)

REQUIRED_COLUMNS = {
    "ts_event_utc", "ts_event_ny", "trade_date_utc", "trade_date_ny",
    "product", "symbol", "active_symbol", "open", "high", "low", "close", "volume",
    "hour_ny", "minute_ny", "session_label", "continuous_segment_id",
    "is_roll_day", "is_pre_roll_day", "is_post_roll_day", "roll_window_flag",
    "roll_window_type", "days_since_roll", "days_to_next_roll", "tradable_research_flag",
}
CORE_OHLCV_COLUMNS = ["open", "high", "low", "close", "volume"]
EXPECTED_ROW_COUNTS = {"GC": 1_759_671, "MGC": 1_727_985}
OUTCOME_NAME_PATTERNS = (
    r"(^|_)forward(_|$)",
    r"(^|_)future(_|$)",
    r"(^|_)(mfe|mae)(_|$)",
    r"(^|_)(target|stop).*(hit|outcome)",
    r"(^|_)outcome(_|$)",
)
EXISTING_FORWARD_COLUMNS = [
    name
    for name in saved_column_names
    if any(re.search(pattern, name.lower()) for pattern in OUTCOME_NAME_PATTERNS)
]

missing_required_columns = sorted(REQUIRED_COLUMNS.difference(research_bars.columns))
product_counts = research_bars["product"].value_counts(sort=False).sort_index()
per_product_timestamp_sorted = {
    product: research_bars.loc[research_bars["product"].eq(product), "ts_event_utc"].is_monotonic_increasing
    for product in EXPECTED_PRODUCTS
}
global_timestamp_sorted = research_bars["ts_event_utc"].is_monotonic_increasing
duplicate_timestamp_product_rows = int(research_bars.duplicated(list(UTC_REFERENCE_KEY_COLUMNS)).sum())
core_ohlcv_missing = research_bars[CORE_OHLCV_COLUMNS].isna().sum()

schema_checks = pd.Series(
    {
        "all_required_columns_present": not missing_required_columns,
        "expected_products_only": set(product_counts.index) == set(EXPECTED_PRODUCTS),
        "expected_product_row_counts": product_counts.to_dict() == EXPECTED_ROW_COUNTS,
        "timestamps_sorted_within_each_product": all(per_product_timestamp_sorted.values()),
        "duplicate_timestamp_product_rows_zero": duplicate_timestamp_product_rows == 0,
        "core_OHLCV_missing_values_zero": int(core_ohlcv_missing.sum()) == 0,
        "existing_forward_columns_quarantined": len(EXISTING_FORWARD_COLUMNS) > 0,
    },
    name="passed",
)

display(product_counts.rename("rows").to_frame())
display(pd.DataFrame({"timestamp_sorted": per_product_timestamp_sorted}))
display(core_ohlcv_missing.rename("missing_values").to_frame())
print(f"Global physical timestamp order (informational): {global_timestamp_sorted}")
print(f"Duplicate (ts_event_utc, product) rows: {duplicate_timestamp_product_rows:,}")
print(f"EXISTING_FORWARD_COLUMNS ({len(EXISTING_FORWARD_COLUMNS)}): {EXISTING_FORWARD_COLUMNS}")
display(schema_checks.to_frame())

assert schema_checks.all(), schema_checks[~schema_checks].index.tolist()

,column,saved_arrow_type,loaded_pandas_type
0,ts_event_utc,"timestamp[ns, tz=UTC]","timestamp[ns, tz=UTC][pyarrow]"
1,product,large_string,large_string[pyarrow]
2,symbol,large_string,large_string[pyarrow]
3,rtype,uint8,uint8[pyarrow]
4,publisher_id,uint16,uint16[pyarrow]
5,instrument_id,uint32,uint32[pyarrow]
6,open,double,double[pyarrow]
7,high,double,double[pyarrow]
8,low,double,double[pyarrow]
9,close,double,double[pyarrow]


,rows
product,
GC,1759671
MGC,1727985


,timestamp_sorted
GC,True
MGC,True


,missing_values
open,0
high,0
low,0
close,0
volume,0


Global physical timestamp order (informational): False
Duplicate (ts_event_utc, product) rows: 0
EXISTING_FORWARD_COLUMNS (8): ['forward_return_5m', 'forward_log_return_5m', 'forward_return_15m', 'forward_log_return_15m', 'forward_return_30m', 'forward_log_return_30m', 'forward_return_60m', 'forward_log_return_60m']


,passed
all_required_columns_present,True
expected_products_only,True
expected_product_row_counts,True
timestamps_sorted_within_each_product,True
duplicate_timestamp_product_rows_zero,True
core_OHLCV_missing_values_zero,True
existing_forward_columns_quarantined,True


## 2.3 Confirm Timestamp and Time-Zone Handling

The physical timezone metadata is validated directly from the saved schema. Each New York timestamp must represent the same instant as its UTC reference, minute alignment must be exact, the saved New York calendar date and clock fields must agree with `ts_event_ny`, and UTC timestamps are not reinterpreted as session clock time.

In [7]:
utc_arrow_type = saved_arrow_schema.field("ts_event_utc").type
ny_arrow_type = saved_arrow_schema.field("ts_event_ny").type

timezone_checks = pd.Series(
    {
        "ts_event_utc_is_timezone_aware_UTC": pa.types.is_timestamp(utc_arrow_type) and utc_arrow_type.tz == "UTC",
        "ts_event_ny_is_timezone_aware_New_York": pa.types.is_timestamp(ny_arrow_type) and ny_arrow_type.tz == RESEARCH_TIMEZONE_NAME,
        "UTC_and_New_York_columns_represent_same_instants": bool(
            research_bars["ts_event_ny"].dt.tz_convert(REFERENCE_TIMEZONE_NAME).eq(research_bars["ts_event_utc"]).all()
        ),
        "UTC_timestamps_are_exactly_minute_aligned": bool(
            research_bars["ts_event_utc"].eq(research_bars["ts_event_utc"].dt.floor("min")).all()
        ),
        "New_York_trade_dates_match_local_calendar_dates": bool(
            research_bars["ts_event_ny"].dt.date.eq(research_bars["trade_date_ny"].dt.date).all()
        ),
        "saved_hour_ny_matches_local_timestamp": bool(
            research_bars["ts_event_ny"].dt.hour.eq(research_bars["hour_ny"]).all()
        ),
        "saved_minute_ny_matches_local_timestamp": bool(
            research_bars["ts_event_ny"].dt.minute.eq(research_bars["minute_ny"]).all()
        ),
    },
    name="passed",
)
display(timezone_checks.to_frame())
display(research_bars[["ts_event_utc", "ts_event_ny", "trade_date_ny", "hour_ny", "minute_ny"]].head(3))

assert timezone_checks.all(), timezone_checks[~timezone_checks].index.tolist()

,passed
ts_event_utc_is_timezone_aware_UTC,True
ts_event_ny_is_timezone_aware_New_York,True
UTC_and_New_York_columns_represent_same_instants,True
UTC_timestamps_are_exactly_minute_aligned,True
New_York_trade_dates_match_local_calendar_dates,True
saved_hour_ny_matches_local_timestamp,True
saved_minute_ny_matches_local_timestamp,True


,ts_event_utc,ts_event_ny,trade_date_ny,hour_ny,minute_ny
0,2021-05-24 00:00:00+00:00,2021-05-23 20:00:00-04:00,2021-05-23 00:00:00,20,0
1,2021-05-24 00:01:00+00:00,2021-05-23 20:01:00-04:00,2021-05-23 00:00:00,20,1
2,2021-05-24 00:02:00+00:00,2021-05-23 20:02:00-04:00,2021-05-23 00:00:00,20,2


## 2.4 Confirm Date Coverage

Coverage is summarized separately for GC and MGC using both UTC event timestamps and New York trading dates. The two product histories are required to share the reported endpoints and trading-date count. Ordinary futures closures are not treated as missing data in this coverage check.

In [8]:
coverage_by_product = (
    research_bars.groupby("product", observed=True)
    .agg(
        rows=("product", "size"),
        first_ts_utc=("ts_event_utc", "min"),
        last_ts_utc=("ts_event_utc", "max"),
        first_trade_date_ny=("trade_date_ny", "min"),
        last_trade_date_ny=("trade_date_ny", "max"),
        trading_dates=("trade_date_ny", "nunique"),
    )
    .sort_index()
)
display(coverage_by_product)

expected_coverage = {
    "first_ts_utc": pd.Timestamp("2021-05-24 00:00:00", tz="UTC"),
    "last_ts_utc": pd.Timestamp("2026-05-22 20:59:00", tz="UTC"),
    "first_trade_date_ny": pd.Timestamp("2021-05-23"),
    "last_trade_date_ny": pd.Timestamp("2026-05-22"),
    "trading_dates": 1_556,
}
coverage_checks = pd.Series(
    {
        name: bool(coverage_by_product[name].eq(expected).all())
        for name, expected in expected_coverage.items()
    },
    name="passed",
)
display(coverage_checks.to_frame())

assert coverage_checks.all(), coverage_checks[~coverage_checks].index.tolist()

,rows,first_ts_utc,last_ts_utc,first_trade_date_ny,last_trade_date_ny,trading_dates
product,,,,,,
GC,1759671,2021-05-24 00:00:00+00:00,2026-05-22 20:59:00+00:00,2021-05-23 00:00:00,2026-05-22 00:00:00,1556
MGC,1727985,2021-05-24 00:00:00+00:00,2026-05-22 20:59:00+00:00,2021-05-23 00:00:00,2026-05-22 00:00:00,1556


,passed
first_ts_utc,True
last_ts_utc,True
first_trade_date_ny,True
last_trade_date_ny,True
trading_dates,True


## 2.5 Confirm Rollover and Exclusion Flags

Rollover, boundary, and research-tradability controls are validated as saved inputs rather than reconstructed. Boolean values are summarized by product, rollover categories and transition labels are enumerated, and day-level roll-distance fields are described. Missing roll distances are retained at leading/trailing coverage boundaries, while `roll_transition_label` is expected to be populated only on roll days. No eligibility filter is applied in this section.

In [9]:
ROLLOVER_AND_EXCLUSION_COLUMNS = [
    "is_roll_day",
    "is_pre_roll_day",
    "is_post_roll_day",
    "roll_window_flag",
    "roll_window_type",
    "roll_group_id",
    "days_since_roll",
    "days_to_next_roll",
    "roll_transition_label",
    "low_liquidity_active_day_flag",
    "low_liquidity_warning_flag",
    "tradable_research_flag",
]
BOOLEAN_CONTROL_COLUMNS = [
    "is_roll_day",
    "is_pre_roll_day",
    "is_post_roll_day",
    "roll_window_flag",
    "low_liquidity_active_day_flag",
    "low_liquidity_warning_flag",
    "tradable_research_flag",
]
missing_control_columns = sorted(set(ROLLOVER_AND_EXCLUSION_COLUMNS).difference(research_bars.columns))
control_missing_values = research_bars[ROLLOVER_AND_EXCLUSION_COLUMNS].isna().sum()

boolean_true_counts = (
    research_bars.groupby("product", observed=True)[BOOLEAN_CONTROL_COLUMNS].sum().astype("int64")
)
boolean_total_counts = research_bars.groupby("product", observed=True).size().astype("int64")
boolean_summary_records = []
for product in EXPECTED_PRODUCTS:
    for column in BOOLEAN_CONTROL_COLUMNS:
        true_count = int(boolean_true_counts.loc[product, column])
        total_count = int(boolean_total_counts.loc[product])
        boolean_summary_records.append(
            {
                "product": product,
                "field": column,
                "false_rows": total_count - true_count,
                "true_rows": true_count,
                "true_rate": true_count / total_count,
            }
        )
boolean_control_summary = pd.DataFrame(boolean_summary_records).set_index(["product", "field"])

roll_window_type_summary = (
    research_bars.groupby(["product", "roll_window_type"], observed=True, dropna=False)
    .size()
    .rename("rows")
    .to_frame()
)
roll_transition_summary = (
    research_bars.groupby(["product", "roll_transition_label"], observed=True, dropna=False)
    .size()
    .rename("rows")
    .to_frame()
)
daily_roll_state = research_bars[
    ["product", "trade_date_ny", "roll_group_id", "days_since_roll", "days_to_next_roll"]
].drop_duplicates()
roll_distance_summary = daily_roll_state.groupby("product", observed=True)[
    ["roll_group_id", "days_since_roll", "days_to_next_roll"]
].agg(["min", "median", "max"])

display(boolean_control_summary)
display(roll_window_type_summary)
display(roll_transition_summary)
display(roll_distance_summary)
display(control_missing_values.rename("missing_values").to_frame())

control_checks = pd.Series(
    {
        "all_rollover_and_exclusion_fields_present": not missing_control_columns,
        "required_boolean_and_categorical_controls_nonmissing": int(
            control_missing_values[BOOLEAN_CONTROL_COLUMNS + ["roll_window_type", "roll_group_id"]].sum()
        ) == 0,
        "roll_transition_label_present_exactly_on_roll_days": bool(
            research_bars["roll_transition_label"].notna().eq(research_bars["is_roll_day"]).all()
        ),
        "roll_distance_boundary_missingness_is_explicit": bool(
            control_missing_values[["days_since_roll", "days_to_next_roll"]].gt(0).all()
        ),
        "roll_window_flag_matches_roll_window_type": bool(
            research_bars["roll_window_flag"].eq(research_bars["roll_window_type"].ne("none")).all()
        ),
        "tradable_research_flag_has_true_and_false_rows": set(
            research_bars["tradable_research_flag"].unique().tolist()
        ) == {False, True},
    },
    name="passed",
)
display(control_checks.to_frame())

assert control_checks.all(), control_checks[~control_checks].index.tolist()

false_rows  true_rows  true_rate
product field                                                          
GC      is_roll_day                       1727634      32037   0.018206
        is_pre_roll_day                   1738275      21396   0.012159
        is_post_roll_day                  1735453      24218   0.013763
        roll_window_flag                  1682020      77651   0.044128
        low_liquidity_active_day_flag     1739487      20184   0.011470
        low_liquidity_warning_flag        1739487      20184   0.011470
        tradable_research_flag              83080    1676591   0.952787
MGC     is_roll_day                       1697290      30695   0.017763
        is_pre_roll_day                   1716268      11717   0.006781
        is_post_roll_day                  1701656      26329   0.015237
        roll_window_flag                  1659244      68741   0.039781
        low_liquidity_active_day_flag     1706300      21685   0.012549
        low_liquidity_warning_flag        1706300      21685   0.012549
        tradable_research_flag              78709    1649276   0.954450

rows
product roll_window_type         
GC      none              1682020
        post_roll_day       24218
        pre_roll_day        21396
        roll_day            32037
MGC     none              1659244
        post_roll_day       26329
        pre_roll_day        11717
        roll_day            30695

rows
product roll_transition_label         
GC      GCG2 -> GCJ2              1376
        GCG3 -> GCJ3              1380
        GCG4 -> GCJ4              1376
        GCG5 -> GCJ5              1319
        GCG6 -> GCJ6              1380
        GCJ2 -> GCM2              1371
        GCJ3 -> GCM3              1379
        GCJ4 -> GCM4              1260
        GCJ5 -> GCM5               120
        GCJ6 -> GCM6              1380
        GCM1 -> GCQ1              1260
        GCM2 -> GCQ2              1228
        GCM3 -> GCQ3              1371
        GCM4 -> GCQ4              1260
        GCM5 -> GCQ5              1380
        GCQ1 -> GCZ1              1260
        GCQ2 -> GCZ2              1260
        GCQ3 -> GCZ3              1375
        GCQ4 -> GCZ4              1379
        GCQ5 -> GCZ5              1379
        GCZ1 -> GCG2              1379
        GCZ2 -> GCG3              1379
        GCZ3 -> GCG4              1380
        GCZ4 -> GCG5              1184
        GCZ5 -> GCG6              1222
        <NA>                   1727634
MGC     MGCG2 -> MGCJ2            1317
        MGCG3 -> MGCJ3            1369
        MGCG4 -> MGCJ4            1361
        MGCG5 -> MGCJ5            1320
        MGCG6 -> MGCJ6            1380
        MGCJ2 -> MGCM2            1371
        MGCJ3 -> MGCM3             120
        MGCJ4 -> MGCM4            1256
        MGCJ5 -> MGCM5            1380
        MGCJ6 -> MGCM6            1380
        MGCM1 -> MGCQ1             100
        MGCM2 -> MGCQ2            1179
        MGCM3 -> MGCQ3            1368
        MGCM4 -> MGCQ4            1258
        MGCM5 -> MGCQ5            1260
        MGCQ1 -> MGCZ1            1325
        MGCQ2 -> MGCZ2            1317
        MGCQ3 -> MGCZ3            1365
        MGCQ4 -> MGCZ4            1380
        MGCQ5 -> MGCZ5            1380
        MGCZ1 -> MGCG2            1364
        MGCZ2 -> MGCG3            1362
        MGCZ3 -> MGCG4            1370
        MGCZ4 -> MGCG5            1185
        MGCZ5 -> MGCG6            1228
        <NA>                   1697290

roll_group_id               days_since_roll                days_to_next_roll               
                  min    median max             min    median  max               min    median  max
product                                                                                            
GC                  1 13.000000  26               0 30.500000  104                 0 31.000000  104
MGC                 1 13.000000  26               0 30.500000  103                 0 31.000000  103

,missing_values
is_roll_day,0
is_pre_roll_day,0
is_post_roll_day,0
roll_window_flag,0
roll_window_type,0
roll_group_id,0
days_since_roll,9982
days_to_next_roll,101823
roll_transition_label,3424924
low_liquidity_active_day_flag,0


,passed
all_rollover_and_exclusion_fields_present,True
required_boolean_and_categorical_controls_nonmissing,True
roll_transition_label_present_exactly_on_roll_days,True
roll_distance_boundary_missingness_is_explicit,True
roll_window_flag_matches_roll_window_type,True
tradable_research_flag_has_true_and_false_rows,True


### Section 2 validation summary

The executable summary below combines the schema, product, timestamp, timezone, coverage, missingness, rollover/tradability, and forward-column quarantine gates. `READY` means the trusted dataset is suitable for the next milestone—construction of the eligible observation frame—but that construction is intentionally outside the scope of this notebook increment.

In [10]:
section_2_validation = pd.concat(
    [schema_checks, timezone_checks, coverage_checks, control_checks],
    keys=["schema_and_content", "timestamps_and_timezones", "coverage", "rollover_and_exclusions"],
).rename("passed")
ready_for_eligible_observation_frame = bool(section_2_validation.all())

display(section_2_validation.to_frame())
print(
    "Section 2 status: "
    + (
        "READY — the trusted GC/MGC table is validated for construction of the eligible observation frame."
        if ready_for_eligible_observation_frame
        else "NOT READY — at least one validation gate failed."
    )
)
print(f"Quarantined outcome-derived columns: {EXISTING_FORWARD_COLUMNS}")

assert ready_for_eligible_observation_frame

passed
schema_and_content       all_required_columns_present                          True
                         expected_products_only                                True
                         expected_product_row_counts                           True
                         timestamps_sorted_within_each_product                 True
                         duplicate_timestamp_product_rows_zero                 True
                         core_OHLCV_missing_values_zero                        True
                         existing_forward_columns_quarantined                  True
timestamps_and_timezones ts_event_utc_is_timezone_aware_UTC                    True
                         ts_event_ny_is_timezone_aware_New_York                True
                         UTC_and_New_York_columns_represent_same_instants      True
                         UTC_timestamps_are_exactly_minute_aligned             True
                         New_York_trade_dates_match_local_calendar_dates       True
                         saved_hour_ny_matches_local_timestamp                 True
                         saved_minute_ny_matches_local_timestamp               True
coverage                 first_ts_utc                                          True
                         last_ts_utc                                           True
                         first_trade_date_ny                                   True
                         last_trade_date_ny                                    True
                         trading_dates                                         True
rollover_and_exclusions  all_rollover_and_exclusion_fields_present             True
                         required_boolean_and_categorical_controls_nonmi...    True
                         roll_transition_label_present_exactly_on_roll_days    True
                         roll_distance_boundary_missingness_is_explicit        True
                         roll_window_flag_matches_roll_window_type             True
                         tradable_research_flag_has_true_and_false_rows        True

Section 2 status: READY — the trusted GC/MGC table is validated for construction of the eligible observation frame.
Quarantined outcome-derived columns: ['forward_return_5m', 'forward_log_return_5m', 'forward_return_15m', 'forward_log_return_15m', 'forward_return_30m', 'forward_log_return_30m', 'forward_return_60m', 'forward_log_return_60m']


# 3.0 Construct the Eligible Observation Frame

This section maps each valid completed GC decision bar `t` to the true next one-minute bar `t+1`, whose open will be the theoretical entry in later research. Eligibility is determined from the entry timestamp in `America/New_York` and from both bars' saved contract, segment, rollover, liquidity, and tradability state. No prices, forward outcomes, labels, or statistical features are constructed here.

## 3.1 GC Phase 1 Research Population

Phase 1 uses GC only, while the trusted `research_bars` dataframe remains unchanged and continues to contain MGC. A narrow Arrow-backed working table retains only identifiers, timestamps, contract/segment identity, and the saved rollover, liquidity, and tradability controls needed to construct and audit observation pairs. The actual schema provides `symbol`, `active_symbol`, `instrument_id`, and `continuous_segment_id`, so no boundary fallback is required.

In [11]:
assert ready_for_eligible_observation_frame
assert isinstance(research_bars.index, pd.RangeIndex), "Stable source-row identifiers require the trusted RangeIndex."

SECTION_3_SOURCE_COLUMNS = [
    "ts_event_utc",
    "ts_event_ny",
    "trade_date_ny",
    "product",
    "symbol",
    "active_symbol",
    "instrument_id",
    "continuous_segment_id",
    "tradable_research_flag",
    "is_roll_day",
    "is_pre_roll_day",
    "is_post_roll_day",
    "roll_window_flag",
    "roll_window_type",
    "low_liquidity_active_day_flag",
    "low_liquidity_warning_flag",
]
missing_section_3_columns = sorted(set(SECTION_3_SOURCE_COLUMNS).difference(research_bars.columns))
assert not missing_section_3_columns, missing_section_3_columns

gc_source_mask = research_bars["product"].eq(PRIMARY_RESEARCH_PRODUCT)
gc_source_row_ids = np.flatnonzero(gc_source_mask.to_numpy(dtype=bool))
gc_section_3 = research_bars.loc[gc_source_mask, SECTION_3_SOURCE_COLUMNS].copy()
gc_section_3.insert(0, "source_row_id", pd.array(gc_source_row_ids, dtype="int64[pyarrow]"))
gc_source_rows = len(gc_section_3)

derived_tradable_flag = ~(
    gc_section_3["roll_window_flag"]
    | gc_section_3["low_liquidity_active_day_flag"]
    | gc_section_3["low_liquidity_warning_flag"]
)
gc_population_checks = pd.Series(
    {
        "expected_GC_source_rows": gc_source_rows == EXPECTED_ROW_COUNTS[PRIMARY_RESEARCH_PRODUCT],
        "GC_only": set(gc_section_3["product"].unique().tolist()) == {PRIMARY_RESEARCH_PRODUCT},
        "timestamps_sorted": gc_section_3["ts_event_utc"].is_monotonic_increasing,
        "source_row_ids_unique": gc_section_3["source_row_id"].is_unique,
        "selected_symbol_matches_active_symbol": bool(
            gc_section_3["symbol"].eq(gc_section_3["active_symbol"]).all()
        ),
        "outright_GC_symbol_format_only": bool(
            gc_section_3["symbol"].str.fullmatch(r"GC[A-Z]\d{1,2}").all()
        ),
        "contract_segment_identifiers_complete": bool(
            gc_section_3[["symbol", "active_symbol", "instrument_id", "continuous_segment_id"]]
            .notna()
            .all(axis=None)
        ),
        "tradable_flag_matches_saved_controls": bool(
            gc_section_3["tradable_research_flag"].eq(derived_tradable_flag).all()
        ),
        "trusted_source_still_contains_MGC": "MGC" in set(research_bars["product"].unique().tolist()),
    },
    name="passed",
)
display(pd.Series({"GC_source_rows": gc_source_rows, "working_columns": gc_section_3.shape[1]}, name="value").to_frame())
display(gc_population_checks.to_frame())
assert gc_population_checks.all(), gc_population_checks[~gc_population_checks].index.tolist()

,value
GC_source_rows,1759671
working_columns,17


,passed
expected_GC_source_rows,True
GC_only,True
timestamps_sorted,True
source_row_ids_unique,True
selected_symbol_matches_active_symbol,True
outright_GC_symbol_format_only,True
contract_segment_identifiers_complete,True
tradable_flag_matches_saved_controls,True
trusted_source_still_contains_MGC,True


## 3.2 London Execution Observations

London entry eligibility is the half-open New York-time interval `[03:00, 06:00)`. The helper below uses the timezone-aware `ts_event_ny` clock fields, so 03:00 is included, 06:00 is excluded, and daylight-saving changes are handled by `America/New_York` rather than a fixed UTC offset.

In [12]:
LONDON_START_MINUTE_NY = 3 * 60
LONDON_END_MINUTE_NY = 6 * 60


def minute_of_day_ny(timestamps_ny: pd.Series) -> pd.Series:
    """Return the New York wall-clock minute for timezone-aware timestamps."""
    return timestamps_ny.dt.hour * 60 + timestamps_ny.dt.minute


def london_entry_mask(timestamps_ny: pd.Series) -> pd.Series:
    entry_minute = minute_of_day_ny(timestamps_ny)
    return entry_minute.ge(LONDON_START_MINUTE_NY) & entry_minute.lt(LONDON_END_MINUTE_NY)


london_probe = pd.Series(
    pd.to_datetime(["2024-01-02 02:59", "2024-01-02 03:00", "2024-01-02 05:59", "2024-01-02 06:00"])
    .tz_localize(RESEARCH_TIMEZONE_NAME),
    name="entry_timestamp_ny",
)
london_probe_result = london_entry_mask(london_probe)
assert london_probe_result.tolist() == [False, True, True, False]
display(pd.DataFrame({"entry_timestamp_ny": london_probe, "London_eligible": london_probe_result}))

,entry_timestamp_ny,London_eligible
0,2024-01-02 02:59:00-05:00,False
1,2024-01-02 03:00:00-05:00,True
2,2024-01-02 05:59:00-05:00,True
3,2024-01-02 06:00:00-05:00,False


## 3.3 New York Execution Observations

New York entry eligibility is the half-open interval `[07:00, 12:00)` in New York time. This explicitly excludes the 06:00-07:00 gap and any entry at noon or later. Session labels are assigned later from the entry bar timestamp only.

In [13]:
NEW_YORK_START_MINUTE_NY = 7 * 60
NEW_YORK_END_MINUTE_NY = 12 * 60


def new_york_entry_mask(timestamps_ny: pd.Series) -> pd.Series:
    entry_minute = minute_of_day_ny(timestamps_ny)
    return entry_minute.ge(NEW_YORK_START_MINUTE_NY) & entry_minute.lt(NEW_YORK_END_MINUTE_NY)


new_york_probe = pd.Series(
    pd.to_datetime(["2024-01-02 06:59", "2024-01-02 07:00", "2024-01-02 11:59", "2024-01-02 12:00"])
    .tz_localize(RESEARCH_TIMEZONE_NAME),
    name="entry_timestamp_ny",
)
new_york_probe_result = new_york_entry_mask(new_york_probe)
assert new_york_probe_result.tolist() == [False, True, True, False]
display(pd.DataFrame({"entry_timestamp_ny": new_york_probe, "New_York_eligible": new_york_probe_result}))

,entry_timestamp_ny,New_York_eligible
0,2024-01-02 06:59:00-05:00,False
1,2024-01-02 07:00:00-05:00,True
2,2024-01-02 11:59:00-05:00,True
3,2024-01-02 12:00:00-05:00,False


## 3.4 Next-Bar Entry Convention

Each GC row is treated as a candidate completed decision bar and paired vectorially with the next chronological GC row. A pair is structurally valid only when the entry row exists, all required identifiers are present, the UTC gap is exactly one minute, and product, active contract, instrument, continuous segment, and New York trading date remain unchanged. Both bars must also pass the saved tradability and boundary controls.

In [14]:
PAIR_REQUIRED_COLUMNS = [
    "source_row_id",
    "ts_event_utc",
    "ts_event_ny",
    "trade_date_ny",
    "product",
    "symbol",
    "active_symbol",
    "instrument_id",
    "continuous_segment_id",
    "tradable_research_flag",
    "roll_window_flag",
    "roll_window_type",
    "low_liquidity_active_day_flag",
    "low_liquidity_warning_flag",
]
gc_entry_bar = gc_section_3.shift(-1).add_prefix("entry_")

pair_required_present = (
    gc_section_3[PAIR_REQUIRED_COLUMNS].notna().all(axis=1)
    & gc_entry_bar[[f"entry_{column}" for column in PAIR_REQUIRED_COLUMNS]].notna().all(axis=1)
)
decision_entry_gap = gc_entry_bar["entry_ts_event_utc"] - gc_section_3["ts_event_utc"]
same_contract_and_instrument = (
    gc_section_3["symbol"].eq(gc_entry_bar["entry_symbol"])
    & gc_section_3["active_symbol"].eq(gc_entry_bar["entry_active_symbol"])
    & gc_section_3["instrument_id"].eq(gc_entry_bar["entry_instrument_id"])
)
both_tradable = (
    gc_section_3["tradable_research_flag"]
    & gc_entry_bar["entry_tradable_research_flag"]
)
roll_controls_clear = (
    ~gc_section_3["roll_window_flag"]
    & ~gc_entry_bar["entry_roll_window_flag"]
    & gc_section_3["roll_window_type"].eq("none")
    & gc_entry_bar["entry_roll_window_type"].eq("none")
)
liquidity_controls_clear = (
    ~gc_section_3["low_liquidity_active_day_flag"]
    & ~gc_entry_bar["entry_low_liquidity_active_day_flag"]
    & ~gc_section_3["low_liquidity_warning_flag"]
    & ~gc_entry_bar["entry_low_liquidity_warning_flag"]
)
pair_rule_masks = {
    "next_GC_row_available": gc_entry_bar["entry_source_row_id"].notna(),
    "required_pair_fields_present": pair_required_present,
    "exact_one_minute_gap": decision_entry_gap.eq(pd.Timedelta(minutes=1)),
    "same_product": gc_section_3["product"].eq(gc_entry_bar["entry_product"]),
    "same_active_contract_and_instrument": same_contract_and_instrument,
    "same_continuous_segment": gc_section_3["continuous_segment_id"].eq(
        gc_entry_bar["entry_continuous_segment_id"]
    ),
    "same_New_York_trade_date": gc_section_3["trade_date_ny"].eq(gc_entry_bar["entry_trade_date_ny"]),
    "both_bars_tradable": both_tradable,
    "roll_controls_clear": roll_controls_clear,
    "liquidity_controls_clear": liquidity_controls_clear,
}

display(
    pd.Series(
        {name: int(mask.fillna(False).sum()) for name, mask in pair_rule_masks.items()},
        name="rows_passing_rule_independently",
    ).to_frame()
)

,rows_passing_rule_independently
next_GC_row_available,1759670
required_pair_fields_present,1759670
exact_one_minute_gap,1751042
same_product,1759670
same_active_contract_and_instrument,1759645
same_continuous_segment,1751038
same_New_York_trade_date,1758115
both_bars_tradable,1676565
roll_controls_clear,1681994
liquidity_controls_clear,1739466


## 3.5 Entry Cutoff Rules

The entry bar's New York timestamp determines both cutoff eligibility and session. The sequential funnel applies each rule only to rows surviving prior stages, so every exclusion is counted once and the final row count reconciles exactly to the GC source population. Entries at or after 12:00 are removed before the approved-window rule; the remaining off-window exclusions cover pre-03:00 time and the 06:00-07:00 gap.

In [15]:
entry_minute_of_day_ny = minute_of_day_ny(gc_entry_bar["entry_ts_event_ny"])
entry_before_noon = entry_minute_of_day_ny.lt(NEW_YORK_END_MINUTE_NY)
london_entry_candidate = london_entry_mask(gc_entry_bar["entry_ts_event_ny"])
new_york_entry_candidate = new_york_entry_mask(gc_entry_bar["entry_ts_event_ny"])
approved_entry_window = london_entry_candidate | new_york_entry_candidate

funnel_rules = [
    ("Next GC row available", pair_rule_masks["next_GC_row_available"]),
    ("Required pair fields present", pair_rule_masks["required_pair_fields_present"]),
    ("Exact consecutive one-minute pair", pair_rule_masks["exact_one_minute_gap"]),
    ("Same product", pair_rule_masks["same_product"]),
    ("Same active contract and instrument", pair_rule_masks["same_active_contract_and_instrument"]),
    ("Same continuous segment", pair_rule_masks["same_continuous_segment"]),
    ("Same New York trading date", pair_rule_masks["same_New_York_trade_date"]),
    ("Both bars tradable", pair_rule_masks["both_bars_tradable"]),
    ("Roll controls clear", pair_rule_masks["roll_controls_clear"]),
    ("Liquidity controls clear", pair_rule_masks["liquidity_controls_clear"]),
    ("Entry before noon NY", entry_before_noon),
    ("Entry in approved London or New York window", approved_entry_window),
]
remaining_mask = pd.Series(True, index=gc_section_3.index, dtype=bool)
funnel_records = [
    {"stage": "GC source decision rows", "excluded_at_stage": 0, "remaining_rows": gc_source_rows}
]
for stage, condition in funnel_rules:
    rows_before = int(remaining_mask.sum())
    remaining_mask &= condition.fillna(False)
    rows_after = int(remaining_mask.sum())
    funnel_records.append(
        {"stage": stage, "excluded_at_stage": rows_before - rows_after, "remaining_rows": rows_after}
    )

eligible_decision_mask = remaining_mask
exclusion_funnel = pd.DataFrame(funnel_records)
eligible_observation_count = int(eligible_decision_mask.sum())
assert int(exclusion_funnel["excluded_at_stage"].sum()) + eligible_observation_count == gc_source_rows
assert eligible_observation_count > 0
display(exclusion_funnel)

,stage,excluded_at_stage,remaining_rows
0,GC source decision rows,0,1759671
1,Next GC row available,1,1759670
2,Required pair fields present,0,1759670
3,Exact consecutive one-minute pair,8628,1751042
4,Same product,0,1751042
5,Same active contract and instrument,4,1751038
6,Same continuous segment,0,1751038
7,Same New York trading date,1271,1749767
8,Both bars tradable,79014,1670753
9,Roll controls clear,0,1670753


## 3.6 End-of-Day Exit Constraints

The compact observation index is created only for rows surviving the complete funnel. The mandatory exit is localized from the saved New York trading date at 15:30 `America/New_York`; `minutes_to_forced_exit` is schedule metadata only and does not inspect future bars or prices. Frozen research partitions are assigned from the New York entry date without using them to make any selection decision.

In [16]:
eligible_index = gc_section_3.index[eligible_decision_mask]


def decision_values(column: str) -> pd.Series:
    return gc_section_3.loc[eligible_index, column].reset_index(drop=True)


def entry_values(column: str) -> pd.Series:
    return gc_entry_bar.loc[eligible_index, f"entry_{column}"].reset_index(drop=True)


eligible_observations_gc = pd.DataFrame(
    {
        "observation_id": pd.array(np.arange(eligible_observation_count), dtype="int64[pyarrow]"),
        "decision_bar_id": decision_values("source_row_id").astype("int64[pyarrow]"),
        "entry_bar_id": entry_values("source_row_id").astype("int64[pyarrow]"),
        "decision_timestamp_utc": decision_values("ts_event_utc"),
        "decision_timestamp_ny": decision_values("ts_event_ny"),
        "entry_timestamp_utc": entry_values("ts_event_utc"),
        "entry_timestamp_ny": entry_values("ts_event_ny"),
        "trade_date_ny": entry_values("trade_date_ny"),
        "product": decision_values("product"),
        "symbol": decision_values("symbol"),
        "active_symbol": decision_values("active_symbol"),
        "instrument_id": decision_values("instrument_id"),
        "continuous_segment_id": decision_values("continuous_segment_id"),
        "decision_tradable_flag": decision_values("tradable_research_flag"),
        "entry_tradable_flag": entry_values("tradable_research_flag"),
        "decision_roll_window_flag": decision_values("roll_window_flag"),
        "entry_roll_window_flag": entry_values("roll_window_flag"),
        "decision_roll_window_type": decision_values("roll_window_type"),
        "entry_roll_window_type": entry_values("roll_window_type"),
        "decision_is_roll_day": decision_values("is_roll_day"),
        "entry_is_roll_day": entry_values("is_roll_day"),
        "decision_is_pre_roll_day": decision_values("is_pre_roll_day"),
        "entry_is_pre_roll_day": entry_values("is_pre_roll_day"),
        "decision_is_post_roll_day": decision_values("is_post_roll_day"),
        "entry_is_post_roll_day": entry_values("is_post_roll_day"),
        "decision_low_liquidity_flag": decision_values("low_liquidity_warning_flag"),
        "entry_low_liquidity_flag": entry_values("low_liquidity_warning_flag"),
    }
)
eligible_observations_gc["entry_hour_ny"] = (
    eligible_observations_gc["entry_timestamp_ny"].dt.hour.astype("int8[pyarrow]")
)
eligible_observations_gc["entry_minute_ny"] = (
    eligible_observations_gc["entry_timestamp_ny"].dt.minute.astype("int8[pyarrow]")
)
eligible_entry_minute = (
    eligible_observations_gc["entry_hour_ny"] * 60 + eligible_observations_gc["entry_minute_ny"]
)
eligible_observations_gc["entry_session"] = pd.Categorical(
    np.where(eligible_entry_minute.lt(LONDON_END_MINUTE_NY), "London", "New York"),
    categories=["London", "New York"],
    ordered=True,
)
eligible_observations_gc["entry_year"] = (
    eligible_observations_gc["entry_timestamp_ny"].dt.year.astype("int16[pyarrow]")
)
entry_date_ny = eligible_observations_gc["trade_date_ny"]
partition_labels = np.select(
    [
        entry_date_ny.le(pd.Timestamp("2023-12-31")).to_numpy(dtype=bool),
        entry_date_ny.between(pd.Timestamp("2024-01-01"), pd.Timestamp("2024-12-31")).to_numpy(dtype=bool),
        entry_date_ny.between(pd.Timestamp("2025-01-01"), pd.Timestamp("2026-05-22")).to_numpy(dtype=bool),
    ],
    ["Development", "Validation", "Final test"],
    default="UNASSIGNED",
)
eligible_observations_gc["research_partition"] = pd.Categorical(
    partition_labels,
    categories=["Development", "Validation", "Final test"],
    ordered=True,
)
forced_exit_naive = eligible_observations_gc["trade_date_ny"] + pd.Timedelta(hours=15, minutes=30)
eligible_observations_gc["forced_exit_timestamp_ny"] = forced_exit_naive.dt.tz_localize(
    RESEARCH_TIMEZONE_NAME, ambiguous="raise", nonexistent="raise"
)
minutes_to_exit_float = (
    eligible_observations_gc["forced_exit_timestamp_ny"] - eligible_observations_gc["entry_timestamp_ny"]
).dt.total_seconds() / 60
assert np.allclose(minutes_to_exit_float.to_numpy(dtype=float), np.rint(minutes_to_exit_float.to_numpy(dtype=float)))
eligible_observations_gc["minutes_to_forced_exit"] = pd.array(
    np.rint(minutes_to_exit_float.to_numpy(dtype=float)).astype(np.int16), dtype="int16[pyarrow]"
)

OUTPUT_COLUMN_ORDER = [
    "observation_id", "decision_bar_id", "entry_bar_id",
    "decision_timestamp_utc", "decision_timestamp_ny", "entry_timestamp_utc", "entry_timestamp_ny",
    "trade_date_ny", "entry_session", "entry_hour_ny", "entry_minute_ny", "entry_year",
    "research_partition", "product", "symbol", "active_symbol", "instrument_id",
    "continuous_segment_id", "decision_tradable_flag", "entry_tradable_flag",
    "decision_roll_window_flag", "entry_roll_window_flag",
    "decision_roll_window_type", "entry_roll_window_type",
    "decision_is_roll_day", "entry_is_roll_day",
    "decision_is_pre_roll_day", "entry_is_pre_roll_day",
    "decision_is_post_roll_day", "entry_is_post_roll_day",
    "decision_low_liquidity_flag", "entry_low_liquidity_flag",
    "forced_exit_timestamp_ny", "minutes_to_forced_exit",
]
eligible_observations_gc = eligible_observations_gc.loc[:, OUTPUT_COLUMN_ORDER]
display(eligible_observations_gc.head(3))

,observation_id,decision_bar_id,entry_bar_id,decision_timestamp_utc,decision_timestamp_ny,entry_timestamp_utc,entry_timestamp_ny,trade_date_ny,entry_session,entry_hour_ny,entry_minute_ny,entry_year,research_partition,product,symbol,active_symbol,instrument_id,continuous_segment_id,decision_tradable_flag,entry_tradable_flag,decision_roll_window_flag,entry_roll_window_flag,decision_roll_window_type,entry_roll_window_type,decision_is_roll_day,entry_is_roll_day,decision_is_pre_roll_day,entry_is_pre_roll_day,decision_is_post_roll_day,entry_is_post_roll_day,decision_low_liquidity_flag,entry_low_liquidity_flag,forced_exit_timestamp_ny,minutes_to_forced_exit
0,0,419,420,2021-05-24 06:59:00+00:00,2021-05-24 02:59:00-04:00,2021-05-24 07:00:00+00:00,2021-05-24 03:00:00-04:00,2021-05-24 00:00:00,London,3,0,2021,Development,GC,GCM1,GCM1,100440,1,True,True,False,False,none,none,False,False,False,False,False,False,False,False,2021-05-24 15:30:00-04:00,750
1,1,420,421,2021-05-24 07:00:00+00:00,2021-05-24 03:00:00-04:00,2021-05-24 07:01:00+00:00,2021-05-24 03:01:00-04:00,2021-05-24 00:00:00,London,3,1,2021,Development,GC,GCM1,GCM1,100440,1,True,True,False,False,none,none,False,False,False,False,False,False,False,False,2021-05-24 15:30:00-04:00,749
2,2,421,422,2021-05-24 07:01:00+00:00,2021-05-24 03:01:00-04:00,2021-05-24 07:02:00+00:00,2021-05-24 03:02:00-04:00,2021-05-24 00:00:00,London,3,2,2021,Development,GC,GCM1,GCM1,100440,1,True,True,False,False,none,none,False,False,False,False,False,False,False,False,2021-05-24 15:30:00-04:00,748


## 3.7 Missing-Bar, Contract, Segment, and Abnormal-Day Handling

Production checks verify all locked boundaries and summarize the final population by session, year, and frozen partition. A separate synthetic table exercises the specified accept/reject edge cases without modifying production data. The saved `tradable_research_flag` exactly incorporates roll-window and liquidity exclusions; these component flags are still checked explicitly and retained for auditability.

In [17]:
session_counts = eligible_observations_gc["entry_session"].value_counts(sort=False)
year_counts = eligible_observations_gc["entry_year"].value_counts(sort=False).sort_index()
partition_counts = eligible_observations_gc["research_partition"].value_counts(sort=False)
saved_field_missingness = eligible_observations_gc.isna().sum()

important_boundary_minutes = [(2, 59), (3, 0), (5, 59), (6, 0), (6, 59), (7, 0), (11, 59), (12, 0)]
boundary_minute_counts = pd.DataFrame(
    [
        {
            "entry_time_ny": f"{hour:02d}:{minute:02d}",
            "eligible_entries": int(
                (
                    eligible_observations_gc["entry_hour_ny"].eq(hour)
                    & eligible_observations_gc["entry_minute_ny"].eq(minute)
                ).sum()
            ),
        }
        for hour, minute in important_boundary_minutes
    ]
)

eligible_pair_rule_checks = {
    name: bool(mask.loc[eligible_index].all()) for name, mask in pair_rule_masks.items()
}
production_validation_checks = pd.Series(
    {
        "expected_eligible_row_count": len(eligible_observations_gc) == eligible_observation_count,
        "all_pair_rules_pass": all(eligible_pair_rule_checks.values()),
        "entry_strictly_after_decision": bool(
            eligible_observations_gc["entry_timestamp_utc"].gt(
                eligible_observations_gc["decision_timestamp_utc"]
            ).all()
        ),
        "decision_entry_gap_exactly_one_minute": bool(
            eligible_observations_gc["entry_timestamp_utc"]
            .sub(eligible_observations_gc["decision_timestamp_utc"])
            .eq(pd.Timedelta(minutes=1))
            .all()
        ),
        "entry_timestamps_sorted": eligible_observations_gc["entry_timestamp_utc"].is_monotonic_increasing,
        "London_window_exact": bool(
            eligible_entry_minute[eligible_observations_gc["entry_session"].eq("London")]
            .between(LONDON_START_MINUTE_NY, LONDON_END_MINUTE_NY - 1)
            .all()
        ),
        "New_York_window_exact": bool(
            eligible_entry_minute[eligible_observations_gc["entry_session"].eq("New York")]
            .between(NEW_YORK_START_MINUTE_NY, NEW_YORK_END_MINUTE_NY - 1)
            .all()
        ),
        "no_entries_during_06_to_07_gap": bool(
            ~eligible_entry_minute.between(LONDON_END_MINUTE_NY, NEW_YORK_START_MINUTE_NY - 1).any()
        ),
        "no_entries_at_or_after_noon": bool(eligible_entry_minute.lt(NEW_YORK_END_MINUTE_NY).all()),
        "same_New_York_date_as_forced_exit": bool(
            eligible_observations_gc["forced_exit_timestamp_ny"].dt.date.eq(
                eligible_observations_gc["trade_date_ny"].dt.date
            ).all()
        ),
        "minutes_to_forced_exit_positive": bool(
            eligible_observations_gc["minutes_to_forced_exit"].gt(0).all()
        ),
        "observation_ids_unique": eligible_observations_gc["observation_id"].is_unique,
        "decision_entry_pairs_unique": not eligible_observations_gc.duplicated(
            ["decision_bar_id", "entry_bar_id"]
        ).any(),
        "all_saved_fields_complete": int(saved_field_missingness.sum()) == 0,
        "forward_columns_absent": not set(EXISTING_FORWARD_COLUMNS).intersection(
            eligible_observations_gc.columns
        ),
        "all_partitions_assigned": "UNASSIGNED" not in set(partition_labels),
        "all_decision_and_entry_bars_tradable": bool(
            eligible_observations_gc[["decision_tradable_flag", "entry_tradable_flag"]]
            .all(axis=None)
        ),
        "all_roll_and_liquidity_controls_clear": bool(
            ~eligible_observations_gc[
                ["decision_roll_window_flag", "entry_roll_window_flag",
                 "decision_low_liquidity_flag", "entry_low_liquidity_flag"]
            ].any(axis=None)
        ),
    },
    name="passed",
)

display(pd.Series({"GC_source_rows": gc_source_rows, "eligible_observations": len(eligible_observations_gc),
                   "first_entry_ny": eligible_observations_gc["entry_timestamp_ny"].min(),
                   "last_entry_ny": eligible_observations_gc["entry_timestamp_ny"].max()}, name="value").to_frame())
display(session_counts.rename("eligible_observations").to_frame())
display(year_counts.rename("eligible_observations").to_frame())
display(partition_counts.rename("eligible_observations").to_frame())
display(boundary_minute_counts)
display(saved_field_missingness.rename("missing_values").to_frame())
display(production_validation_checks.to_frame())
assert production_validation_checks.all(), production_validation_checks[~production_validation_checks].index.tolist()

C:\Users\abond\AppData\Local\Temp\ipykernel_39224\147277059.py:52: DeprecationWarning: Bitwise inversion '~' on bool is deprecated and will be removed in Python 3.16. This returns the bitwise inversion of the underlying int object and is usually not what you expect from negating a bool. Use the 'not' operator for boolean negation or ~int(x) if you really want the bitwise inversion of the underlying int.
  ~eligible_entry_minute.between(LONDON_END_MINUTE_NY, NEW_YORK_START_MINUTE_NY - 1).any()


,value
GC_source_rows,1759671
eligible_observations,586530
first_entry_ny,2021-05-24 03:00:00-04:00
last_entry_ny,2026-05-22 11:59:00-04:00


,eligible_observations
entry_session,
London,219938
New York,366592


,eligible_observations
entry_year,
2021,72960
2022,117114
2023,116156
2024,118076
2025,117592
2026,44632


,eligible_observations
research_partition,
Development,306230
Validation,118076
Final test,162224


,entry_time_ny,eligible_entries
0,02:59,0
1,03:00,1222
2,05:59,1222
3,06:00,0
4,06:59,0
5,07:00,1222
6,11:59,1222
7,12:00,0


,missing_values
observation_id,0
decision_bar_id,0
entry_bar_id,0
decision_timestamp_utc,0
decision_timestamp_ny,0
entry_timestamp_utc,0
entry_timestamp_ny,0
trade_date_ny,0
entry_session,0
entry_hour_ny,0


,passed
expected_eligible_row_count,True
all_pair_rules_pass,True
entry_strictly_after_decision,True
decision_entry_gap_exactly_one_minute,True
entry_timestamps_sorted,True
London_window_exact,True
New_York_window_exact,True
no_entries_during_06_to_07_gap,True
no_entries_at_or_after_noon,True
same_New_York_date_as_forced_exit,True


In [18]:
synthetic_pairs = pd.DataFrame(
    {
        "case": [
            "02:59 to 03:00", "05:59 to 06:00", "06:59 to 07:00", "11:58 to 11:59",
            "11:59 to 12:00", "contract change", "non-consecutive pair", "New York date crossing",
        ],
        "decision_time": [
            "2024-01-02 02:59", "2024-01-02 05:59", "2024-01-02 06:59", "2024-01-02 11:58",
            "2024-01-02 11:59", "2024-01-02 07:00", "2024-01-02 07:00", "2024-01-02 23:59",
        ],
        "entry_time": [
            "2024-01-02 03:00", "2024-01-02 06:00", "2024-01-02 07:00", "2024-01-02 11:59",
            "2024-01-02 12:00", "2024-01-02 07:01", "2024-01-02 07:02", "2024-01-03 00:00",
        ],
        "decision_symbol": ["GCG4"] * 8,
        "entry_symbol": ["GCG4", "GCG4", "GCG4", "GCG4", "GCG4", "GCJ4", "GCG4", "GCG4"],
        "expected_eligible": [True, False, True, True, False, False, False, False],
    }
)
synthetic_pairs["decision_timestamp_ny"] = pd.to_datetime(synthetic_pairs.pop("decision_time")).dt.tz_localize(
    RESEARCH_TIMEZONE_NAME
)
synthetic_pairs["entry_timestamp_ny"] = pd.to_datetime(synthetic_pairs.pop("entry_time")).dt.tz_localize(
    RESEARCH_TIMEZONE_NAME
)
synthetic_gap_valid = synthetic_pairs["entry_timestamp_ny"].sub(
    synthetic_pairs["decision_timestamp_ny"]
).eq(pd.Timedelta(minutes=1))
synthetic_contract_valid = synthetic_pairs["decision_symbol"].eq(synthetic_pairs["entry_symbol"])
synthetic_date_valid = synthetic_pairs["decision_timestamp_ny"].dt.date.eq(
    synthetic_pairs["entry_timestamp_ny"].dt.date
)
synthetic_window_valid = london_entry_mask(synthetic_pairs["entry_timestamp_ny"]) | new_york_entry_mask(
    synthetic_pairs["entry_timestamp_ny"]
)
synthetic_pairs["actual_eligible"] = (
    synthetic_gap_valid & synthetic_contract_valid & synthetic_date_valid & synthetic_window_valid
)
synthetic_pairs["test_passed"] = synthetic_pairs["actual_eligible"].eq(
    synthetic_pairs["expected_eligible"]
)
display(synthetic_pairs[["case", "decision_timestamp_ny", "entry_timestamp_ny",
                         "expected_eligible", "actual_eligible", "test_passed"]])
assert synthetic_pairs["test_passed"].all()

,case,decision_timestamp_ny,entry_timestamp_ny,expected_eligible,actual_eligible,test_passed
0,02:59 to 03:00,2024-01-02 02:59:00-05:00,2024-01-02 03:00:00-05:00,True,True,True
1,05:59 to 06:00,2024-01-02 05:59:00-05:00,2024-01-02 06:00:00-05:00,False,False,True
2,06:59 to 07:00,2024-01-02 06:59:00-05:00,2024-01-02 07:00:00-05:00,True,True,True
3,11:58 to 11:59,2024-01-02 11:58:00-05:00,2024-01-02 11:59:00-05:00,True,True,True
4,11:59 to 12:00,2024-01-02 11:59:00-05:00,2024-01-02 12:00:00-05:00,False,False,True
5,contract change,2024-01-02 07:00:00-05:00,2024-01-02 07:01:00-05:00,False,False,True
6,non-consecutive pair,2024-01-02 07:00:00-05:00,2024-01-02 07:02:00-05:00,False,False,True
7,New York date crossing,2024-01-02 23:59:00-05:00,2024-01-03 00:00:00-05:00,False,False,True


## 3.8 Save Eligible Observation Index

The validated index is saved under the generated-data directory with Zstandard compression, then reloaded as an independent artifact. Reload checks confirm row and column identity, timestamp timezone preservation, identifier uniqueness, complete saved fields, and continued exclusion of every quarantined forward-looking column. The generated parquet is excluded from Git by the project data ignore rule.

In [19]:
STATISTICAL_RESEARCH_DATA_DIR = PROJECT_ROOT / "data" / "processed" / "statistical_research"
ELIGIBLE_OBSERVATIONS_PATH = STATISTICAL_RESEARCH_DATA_DIR / "eligible_observations_gc.parquet"
STATISTICAL_RESEARCH_DATA_DIR.mkdir(parents=True, exist_ok=True)
eligible_observations_gc.to_parquet(
    ELIGIBLE_OBSERVATIONS_PATH, engine="pyarrow", compression="zstd", index=False
)

reloaded_eligible_observations_gc = pd.read_parquet(
    ELIGIBLE_OBSERVATIONS_PATH, engine="pyarrow", dtype_backend="pyarrow"
)
reloaded_schema = pq.ParquetFile(ELIGIBLE_OBSERVATIONS_PATH).schema_arrow
reload_missingness = reloaded_eligible_observations_gc.isna().sum()
reload_checks = pd.Series(
    {
        "row_count_preserved": len(reloaded_eligible_observations_gc) == len(eligible_observations_gc),
        "column_order_preserved": list(reloaded_eligible_observations_gc.columns) == OUTPUT_COLUMN_ORDER,
        "decision_UTC_timezone_preserved": reloaded_schema.field("decision_timestamp_utc").type.tz == "UTC",
        "entry_UTC_timezone_preserved": reloaded_schema.field("entry_timestamp_utc").type.tz == "UTC",
        "decision_New_York_timezone_preserved": (
            reloaded_schema.field("decision_timestamp_ny").type.tz == RESEARCH_TIMEZONE_NAME
        ),
        "entry_New_York_timezone_preserved": (
            reloaded_schema.field("entry_timestamp_ny").type.tz == RESEARCH_TIMEZONE_NAME
        ),
        "forced_exit_New_York_timezone_preserved": (
            reloaded_schema.field("forced_exit_timestamp_ny").type.tz == RESEARCH_TIMEZONE_NAME
        ),
        "observation_ids_unique_after_reload": reloaded_eligible_observations_gc["observation_id"].is_unique,
        "decision_entry_pairs_unique_after_reload": not reloaded_eligible_observations_gc.duplicated(
            ["decision_bar_id", "entry_bar_id"]
        ).any(),
        "no_missing_saved_values_after_reload": int(reload_missingness.sum()) == 0,
        "no_quarantined_forward_columns_saved": not set(EXISTING_FORWARD_COLUMNS).intersection(
            reloaded_eligible_observations_gc.columns
        ),
    },
    name="passed",
)
display(pd.DataFrame({"column": reloaded_schema.names,
                      "saved_arrow_type": [str(field.type) for field in reloaded_schema]}))
display(reload_checks.to_frame())
assert reload_checks.all(), reload_checks[~reload_checks].index.tolist()

eligible_observations_gc = reloaded_eligible_observations_gc
section_3_ready = bool(production_validation_checks.all() and synthetic_pairs["test_passed"].all() and reload_checks.all())
print(f"Final eligibility: valid GC decision bar t mapped to the true one-minute entry bar t+1; both bars are "
      f"tradable, roll/liquidity clear, contract/instrument/segment/date stable, with entry in [03:00, 06:00) "
      f"or [07:00, 12:00) America/New_York.")
print(f"Eligible observations: {len(eligible_observations_gc):,}")
print(f"Session split: {session_counts.to_dict()}")
print(f"Partition split: {partition_counts.to_dict()}")
print(f"Saved output: {ELIGIBLE_OBSERVATIONS_PATH.relative_to(PROJECT_ROOT)}")
print("Schema fallback: none; symbol, active_symbol, instrument_id, and continuous_segment_id were available.")
print("Section 4 is ready to begin, but no Section 4 work is performed here.")
print("SECTION 3 STATUS: READY" if section_3_ready else "SECTION 3 STATUS: NOT READY")
assert section_3_ready

del gc_entry_bar, remaining_mask, eligible_decision_mask, forced_exit_naive, minutes_to_exit_float

,column,saved_arrow_type
0,observation_id,int64
1,decision_bar_id,int64
2,entry_bar_id,int64
3,decision_timestamp_utc,"timestamp[ns, tz=UTC]"
4,decision_timestamp_ny,"timestamp[ns, tz=America/New_York]"
5,entry_timestamp_utc,"timestamp[ns, tz=UTC]"
6,entry_timestamp_ny,"timestamp[ns, tz=America/New_York]"
7,trade_date_ny,timestamp[ms]
8,entry_session,"dictionary<values=string, indices=int8, ordere..."
9,entry_hour_ny,int8


,passed
row_count_preserved,True
column_order_preserved,True
decision_UTC_timezone_preserved,True
entry_UTC_timezone_preserved,True
decision_New_York_timezone_preserved,True
entry_New_York_timezone_preserved,True
forced_exit_New_York_timezone_preserved,True
observation_ids_unique_after_reload,True
decision_entry_pairs_unique_after_reload,True
no_missing_saved_values_after_reload,True


Final eligibility: valid GC decision bar t mapped to the true one-minute entry bar t+1; both bars are tradable, roll/liquidity clear, contract/instrument/segment/date stable, with entry in [03:00, 06:00) or [07:00, 12:00) America/New_York.
Eligible observations: 586,530
Session split: {'London': 219938, 'New York': 366592}
Partition split: {'Development': 306230, 'Validation': 118076, 'Final test': 162224}
Saved output: data\processed\statistical_research\eligible_observations_gc.parquet
Schema fallback: none; symbol, active_symbol, instrument_id, and continuous_segment_id were available.
Section 4 is ready to begin, but no Section 4 work is performed here.
SECTION 3 STATUS: READY
